# Talos v1 — Kaggle Training Runner

Cycles through 4 phases with automatic checkpoint resume.
Mounts Google Drive for shared checkpoint storage across accounts.

In [ ]:
import os, sys, shutil, subprocess, time, glob, json
from pathlib import Path

# ── Config ──────────────────────────────────────────────────
DATASET = "your-kaggle-username/talos-training"  # CHANGE ME
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/Talos_Checkpoints"
LOCAL_DIR = "/kaggle/working/talos"
BINARY = "Talos"
REPLAY_FILE = "serialized_replays.bin"
DEVICE = "cuda"
NUM_GAMES = 20
CHECKPOINT_DIR = f"{LOCAL_DIR}/checkpoints"
PHASE_FILE = f"{CHECKPOINT_DIR}/current_phase.txt"
S3M_STEPS = 5_000_000  # checkpoint interval

os.makedirs(LOCAL_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [ ]:
# ── Download Dataset ────────────────────────────────────────
try:
    import kagglehub
    path = kagglehub.dataset_download(DATASET)
    print(f"Dataset downloaded to {path}")
    # Copy files to working directory
    for item in os.listdir(path):
        src = os.path.join(path, item)
        dst = os.path.join(LOCAL_DIR, item)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)
except Exception as e:
    print(f"kagglehub failed: {e}, trying direct copy...")
    # Fallback: assume dataset is pre-cached in /kaggle/input
    input_dir = f"/kaggle/input/{DATASET.split('/')[-1]}"
    if os.path.exists(input_dir):
        shutil.copytree(input_dir, LOCAL_DIR, dirs_exist_ok=True)
    else:
        print("No dataset found!")
        raise

In [ ]:
# ── Mount Google Drive (for shared checkpoints) ────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
print("Drive mounted")

In [ ]:
# ── Determine Phase & Resume ───────────────────────────────
PHASES = [
    (0, 30_000_000_000, 0.990, 0.05),
    (30_000_000_000, 80_000_000_000, 0.993, 0.03),
    (80_000_000_000, 150_000_000_000, 0.995, 0.02),
    (150_000_000_000, 200_000_000_000, 0.998, 0.01),
]

def get_latest_checkpoint(ckpt_dir):
    dirs = [d for d in os.listdir(ckpt_dir) if os.path.isdir(os.path.join(ckpt_dir, d)) and d.isdigit()]
    if not dirs:
        return None
    latest = max(int(d) for d in dirs)
    return f"{ckpt_dir}/{latest}"

# Sync checkpoints from Drive → local
print("Syncing checkpoints from Drive...")
for item in os.listdir(DRIVE_CHECKPOINT_DIR):
    src = os.path.join(DRIVE_CHECKPOINT_DIR, item)
    dst = os.path.join(CHECKPOINT_DIR, item)
    if os.path.isdir(src) and not os.path.exists(dst):
        shutil.copytree(src, dst)
    elif os.path.isfile(src):
        shutil.copy2(src, dst)

# Determine phase from total timesteps
latest_cp = get_latest_checkpoint(CHECKPOINT_DIR)
if latest_cp:
    total_steps = int(os.path.basename(latest_cp))
    print(f"Resuming from checkpoint at {total_steps} steps")
else:
    total_steps = 0
    print("No checkpoint found, starting fresh")

phase_idx = 0
for i, (start, end, _, _) in enumerate(PHASES):
    if start <= total_steps < end:
        phase_idx = i
        break
if total_steps >= PHASES[-1][1]:
    phase_idx = 3

print(f"Phase {phase_idx + 1}/4 ({total_steps} total steps)")

In [ ]:
# ── Build Command ───────────────────────────────────────────
binary_path = shutil.which(BINARY, path=LOCAL_DIR) or os.path.join(LOCAL_DIR, BINARY)
if not os.path.exists(binary_path):
    print(f"Binary not found at {binary_path}, trying ./{BINARY}")
    binary_path = f"./{BINARY}"

cmd = [
    binary_path,
    "--device", DEVICE,
    "--games", str(NUM_GAMES),
    "--phase", str(phase_idx),
    "--save-dir", CHECKPOINT_DIR,
]

replay_file = os.path.join(LOCAL_DIR, REPLAY_FILE)
if os.path.exists(replay_file):
    cmd += ["--replays", replay_file]
    print(f"Replay file found: {replay_file}")
else:
    print("No replay file found, using procedural states only")

if latest_cp:
    cmd += ["--resume", CHECKPOINT_DIR]

print(f"Running: {' '.join(cmd)}")

In [ ]:
# ── Run Binary ────────────────────────────────────────────
# Change to LOCAL_DIR so collision_meshes/ is found
os.chdir(LOCAL_DIR)

# Use Popen to capture stdout in real-time
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           universal_newlines=True, bufsize=1)

try:
    for line in process.stdout:
        print(line, end='')
except KeyboardInterrupt:
    print("KeyboardInterrupt, terminating...")
    process.terminate()

process.wait()
print(f"Process exited with code {process.returncode}")

In [ ]:
# ── Upload Checkpoints to Drive ───────────────────────────
print("Uploading checkpoints to Drive...")
for item in os.listdir(CHECKPOINT_DIR):
    src = os.path.join(CHECKPOINT_DIR, item)
    dst = os.path.join(DRIVE_CHECKPOINT_DIR, item)
    if os.path.isdir(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
    elif os.path.isfile(src):
        shutil.copy2(src, dst)
print("Upload complete")
print("Done!")